In [5]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import norm


# ============================================================
# 1. 读取数据
# ============================================================

file_path = r"D:\作业合集\数字化市场分析\论文\数据\data.xlsx"

data = pd.read_excel(
    file_path,
    sheet_name="Total"
)

print("Data shape:", data.shape)
print(data.head())


# ============================================================
# 2. 定义变量
# ============================================================

X = "Community"
Y = "Evaluating AI"

M1 = "Access Motivation"   # Value Efficacy
M2 = "Skill Motivation"    # Skill Efficacy
M3 = "Usage Motivation"    # Usage Efficacy

controls = ["Gender", "Year"]

variables = [
    X, Y, M1, M2, M3,
    "Gender", "Year"
]

# 只保留分析需要的变量
df = data[variables].copy()

# 删除缺失值
df = df.dropna().reset_index(drop=True)

print("\nFinal N:", len(df))


# ============================================================
# 3. 检查变量类型
# ============================================================

print("\nVariable types:")
print(df.dtypes)

print("\nCommunity distribution:")
print(df[X].value_counts().sort_index())

print("\nGender distribution:")
print(df["Gender"].value_counts().sort_index())

print("\nYear distribution:")
print(df["Year"].value_counts().sort_index())


# ============================================================
# 4. 如果 Gender / Year 是分类变量，进行 dummy coding
# ============================================================

# Community 如果本身已经是数值编码，则直接使用
# 如果 Community 是字符串，也会自动转换成 category

if not pd.api.types.is_numeric_dtype(df[X]):
    df[X] = pd.Categorical(df[X]).codes

# Gender
if not pd.api.types.is_numeric_dtype(df["Gender"]):
    gender_dummies = pd.get_dummies(
        df["Gender"],
        prefix="Gender",
        drop_first=True,
        dtype=float
    )
else:
    gender_dummies = df[["Gender"]].astype(float)

# Year
if not pd.api.types.is_numeric_dtype(df["Year"]):
    year_dummies = pd.get_dummies(
        df["Year"],
        prefix="Year",
        drop_first=True,
        dtype=float
    )
else:
    year_dummies = df[["Year"]].astype(float)

# 合并控制变量
control_df = pd.concat(
    [gender_dummies, year_dummies],
    axis=1
)

# 确保所有数据都是 numeric
df[X] = pd.to_numeric(df[X])
df[Y] = pd.to_numeric(df[Y])
df[M1] = pd.to_numeric(df[M1])
df[M2] = pd.to_numeric(df[M2])


# ============================================================
# 5. Step 1: Total Effect
#
# Community → Evaluating AI
#
# Y = cX + controls
# ============================================================

X_total = pd.concat(
    [
        df[[X]],
        control_df
    ],
    axis=1
)

X_total = sm.add_constant(X_total)

model_total = sm.OLS(
    df[Y],
    X_total
).fit()

total_effect = model_total.params[X]

print("\n" + "=" * 70)
print("TOTAL EFFECT")
print("=" * 70)

print("Total effect (c):", total_effect)
print("p-value:", model_total.pvalues[X])


# ============================================================
# 6. Step 2: Direct Effect + a paths
#
# M = aX + controls
#
# Y = c'X + bM + controls
# ============================================================

mediators = [M1, M2, M3]

a_paths = {}
b_paths = {}
indirect_effects = {}

# ---------- a paths ----------

for M in mediators:

    X_m = pd.concat(
        [
            df[[X]],
            control_df
        ],
        axis=1
    )

    X_m = sm.add_constant(X_m)

    model_a = sm.OLS(
        df[M],
        X_m
    ).fit()

    a_paths[M] = model_a.params[X]


# ---------- b paths + direct effect ----------

X_direct = pd.concat(
    [
        df[[X, M1, M2, M3]],
        control_df
    ],
    axis=1
)

X_direct = sm.add_constant(X_direct)

model_direct = sm.OLS(
    df[Y],
    X_direct
).fit()

direct_effect = model_direct.params[X]

for M in mediators:
    b_paths[M] = model_direct.params[M]

    indirect_effects[M] = (
        a_paths[M] * b_paths[M]
    )


# ============================================================
# 7. 打印原始点估计
# ============================================================

print("\n" + "=" * 70)
print("MEDIATION RESULTS")
print("=" * 70)

print("\nTotal effect:")
print(f"c = {total_effect:.6f}")

print("\nDirect effect:")
print(f"c' = {direct_effect:.6f}")

print("\nPaths:")

for M in mediators:

    print(
        f"\n{X} → {M}: "
        f"a = {a_paths[M]:.6f}"
    )

    print(
        f"{M} → {Y}: "
        f"b = {b_paths[M]:.6f}"
    )

    print(
        f"Indirect effect = a × b = "
        f"{indirect_effects[M]:.6f}"
    )


# ============================================================
# 8. Bootstrap
# ============================================================

N_BOOT = 5000

rng = np.random.default_rng(42)

boot_total = []
boot_direct = []

boot_indirect = {
    M1: [],
    M2: [],
    M3: []
}

n = len(df)

print("\n" + "=" * 70)
print("BOOTSTRAP")
print("=" * 70)

print(f"Bootstrap samples: {N_BOOT}")

for i in range(N_BOOT):

    # bootstrap sampling
    indices = rng.integers(
        low=0,
        high=n,
        size=n
    )

    boot_df = df.iloc[
        indices
    ].reset_index(drop=True)

    # 控制变量同步抽样
    boot_control_df = control_df.iloc[
        indices
    ].reset_index(drop=True)

    # --------------------------------------------------------
    # Total effect
    # --------------------------------------------------------

    X_t = pd.concat(
        [
            boot_df[[X]],
            boot_control_df
        ],
        axis=1
    )

    X_t = sm.add_constant(X_t)

    try:

        model_t = sm.OLS(
            boot_df[Y],
            X_t
        ).fit()

        c = model_t.params[X]

        boot_total.append(c)

    except:
        continue


    # --------------------------------------------------------
    # a paths
    # --------------------------------------------------------

    boot_a = {}

    try:

        for M in mediators:

            X_a = pd.concat(
                [
                    boot_df[[X]],
                    boot_control_df
                ],
                axis=1
            )

            X_a = sm.add_constant(X_a)

            model_a = sm.OLS(
                boot_df[M],
                X_a
            ).fit()

            boot_a[M] = model_a.params[X]


        # ----------------------------------------------------
        # Direct + b paths
        # ----------------------------------------------------

        X_b = pd.concat(
            [
                boot_df[[X, M1, M2, M3]],
                boot_control_df
            ],
            axis=1
        )

        X_b = sm.add_constant(X_b)

        model_b = sm.OLS(
            boot_df[Y],
            X_b
        ).fit()

        c_prime = model_b.params[X]

        boot_direct.append(c_prime)


        # ----------------------------------------------------
        # indirect effects
        # ----------------------------------------------------

        for M in mediators:

            b = model_b.params[M]

            indirect = (
                boot_a[M] * b
            )

            boot_indirect[M].append(
                indirect
            )

    except:
        continue


# ============================================================
# 9. BCa Bootstrap 95% CI
# ============================================================

def bca_ci(
    bootstrap_values,
    original_estimate,
    data,
    statistic_func,
    alpha=0.05
):
    """
    BCa Bootstrap Confidence Interval

    Parameters
    ----------
    bootstrap_values : array-like
        Bootstrap estimates

    original_estimate : float
        Original sample estimate

    data : DataFrame
        Original dataset

    statistic_func : function
        Function that calculates the statistic from a dataset

    alpha : float
        Significance level, default = 0.05

    Returns
    -------
    lower, upper, z0, acceleration
    """

    bootstrap_values = np.asarray(
        bootstrap_values,
        dtype=float
    )

    bootstrap_values = bootstrap_values[
        np.isfinite(bootstrap_values)
    ]

    # --------------------------------------------------------
    # Step 1: Bias correction z0
    # --------------------------------------------------------

    proportion_less = np.mean(
        bootstrap_values < original_estimate
    )

    # 防止 norm.ppf(0) 或 norm.ppf(1)
    eps = 1e-10

    proportion_less = np.clip(
        proportion_less,
        eps,
        1 - eps
    )

    z0 = norm.ppf(
        proportion_less
    )


    # --------------------------------------------------------
    # Step 2: Jackknife estimates
    # --------------------------------------------------------

    jackknife_values = []

    for i in range(len(data)):

        jackknife_data = data.drop(
            data.index[i]
        ).reset_index(drop=True)

        try:

            estimate = statistic_func(
                jackknife_data
            )

            if np.isfinite(estimate):
                jackknife_values.append(
                    estimate
                )

        except:
            continue

    jackknife_values = np.asarray(
        jackknife_values,
        dtype=float
    )

    jackknife_values = jackknife_values[
        np.isfinite(jackknife_values)
    ]


    # --------------------------------------------------------
    # Step 3: Acceleration parameter
    #
    # a =
    # Σ(θ̄jack - θi)^3
    # --------------------------------
    # 6 [Σ(θ̄jack - θi)^2]^(3/2)
    # --------------------------------------------------------

    jack_mean = np.mean(
        jackknife_values
    )

    numerator = np.sum(
        (jack_mean - jackknife_values) ** 3
    )

    denominator = (
        6 *
        (
            np.sum(
                (jack_mean - jackknife_values) ** 2
            )
            ** 1.5
        )
    )

    if denominator == 0:
        acceleration = 0.0
    else:
        acceleration = (
            numerator / denominator
        )


    # --------------------------------------------------------
    # Step 4: Adjusted alpha levels
    # --------------------------------------------------------

    z_alpha_low = norm.ppf(
        alpha / 2
    )

    z_alpha_high = norm.ppf(
        1 - alpha / 2
    )

    # BCa adjusted probabilities
    adjusted_low = norm.cdf(
        z0
        +
        (
            z0 + z_alpha_low
        )
        /
        (
            1
            -
            acceleration
            *
            (
                z0 + z_alpha_low
            )
        )
    )

    adjusted_high = norm.cdf(
        z0
        +
        (
            z0 + z_alpha_high
        )
        /
        (
            1
            -
            acceleration
            *
            (
                z0 + z_alpha_high
            )
        )
    )

    # 防止概率超出 [0,1]
    adjusted_low = np.clip(
        adjusted_low,
        0,
        1
    )

    adjusted_high = np.clip(
        adjusted_high,
        0,
        1
    )


    # --------------------------------------------------------
    # Step 5: BCa confidence interval
    # --------------------------------------------------------

    lower = np.quantile(
        bootstrap_values,
        adjusted_low
    )

    upper = np.quantile(
        bootstrap_values,
        adjusted_high
    )

    return (
        lower,
        upper,
        z0,
        acceleration
    )


# ============================================================
# 10. 定义每个效应的 statistic function
# ============================================================

def statistic_total(data):

    X_t = pd.concat(
        [
            data[[X]],
            control_df.loc[
                data.index
            ].reset_index(drop=True)
        ],
        axis=1
    )

    X_t = sm.add_constant(X_t)

    model = sm.OLS(
        data[Y].values,
        X_t
    ).fit()

    return model.params[X]


# 为了 jackknife 时控制变量能够同步删除，
# 单独构建完整分析数据
analysis_df = pd.concat(
    [
        df[[X, Y, M1, M2, M3]],
        control_df
    ],
    axis=1
).reset_index(drop=True)


# ------------------------------------------------------------
# Total effect
# ------------------------------------------------------------

def total_statistic(d):

    X_t = d[
        [X] +
        list(control_df.columns)
    ]

    X_t = sm.add_constant(
        X_t,
        has_constant="add"
    )

    model = sm.OLS(
        d[Y],
        X_t
    ).fit()

    return model.params[X]


# ------------------------------------------------------------
# Direct effect
# ------------------------------------------------------------

def direct_statistic(d):

    X_b = d[
        [
            X,
            M1,
            M2,
            M3
        ]
        +
        list(control_df.columns)
    ]

    X_b = sm.add_constant(
        X_b,
        has_constant="add"
    )

    model = sm.OLS(
        d[Y],
        X_b
    ).fit()

    return model.params[X]


# ------------------------------------------------------------
# Indirect effect
# ------------------------------------------------------------

def indirect_statistic(M):

    def statistic(d):

        # a path
        X_a = d[
            [
                X
            ]
            +
            list(control_df.columns)
        ]

        X_a = sm.add_constant(
            X_a,
            has_constant="add"
        )

        model_a = sm.OLS(
            d[M],
            X_a
        ).fit()

        a = model_a.params[X]


        # b path
        X_b = d[
            [
                X,
                M1,
                M2,
                M3
            ]
            +
            list(control_df.columns)
        ]

        X_b = sm.add_constant(
            X_b,
            has_constant="add"
        )

        model_b = sm.OLS(
            d[Y],
            X_b
        ).fit()

        b = model_b.params[M]

        return a * b

    return statistic


# ============================================================
# 11. 计算 BCa CI
# ============================================================

print("\n" + "=" * 70)
print("BCa CONFIDENCE INTERVAL")
print("=" * 70)


# ------------------------------------------------------------
# Total effect
# ------------------------------------------------------------

total_ci = bca_ci(
    bootstrap_values=boot_total,
    original_estimate=total_effect,
    data=analysis_df,
    statistic_func=total_statistic
)

print(
    "\nTotal effect BCa CI:"
)

print(
    f"Estimate = {total_effect:.6f}"
)

print(
    f"95% BCa CI = "
    f"[{total_ci[0]:.6f}, "
    f"{total_ci[1]:.6f}]"
)

print(
    f"Bias correction (z0) = "
    f"{total_ci[2]:.6f}"
)

print(
    f"Acceleration (a) = "
    f"{total_ci[3]:.6f}"
)


# ------------------------------------------------------------
# Direct effect
# ------------------------------------------------------------

direct_ci = bca_ci(
    bootstrap_values=boot_direct,
    original_estimate=direct_effect,
    data=analysis_df,
    statistic_func=direct_statistic
)

print(
    "\nDirect effect BCa CI:"
)

print(
    f"Estimate = {direct_effect:.6f}"
)

print(
    f"95% BCa CI = "
    f"[{direct_ci[0]:.6f}, "
    f"{direct_ci[1]:.6f}]"
)

print(
    f"Bias correction (z0) = "
    f"{direct_ci[2]:.6f}"
)

print(
    f"Acceleration (a) = "
    f"{direct_ci[3]:.6f}"
)


# ------------------------------------------------------------
# Indirect effects
# ------------------------------------------------------------

indirect_ci_results = {}

for M in mediators:

    ci = bca_ci(
        bootstrap_values=boot_indirect[M],
        original_estimate=indirect_effects[M],
        data=analysis_df,
        statistic_func=indirect_statistic(M)
    )

    indirect_ci_results[M] = ci

    print(
        f"\n{M}"
    )

    print(
        f"Indirect effect = "
        f"{indirect_effects[M]:.6f}"
    )

    print(
        f"95% BCa CI = "
        f"[{ci[0]:.6f}, "
        f"{ci[1]:.6f}]"
    )

    print(
        f"Bias correction (z0) = "
        f"{ci[2]:.6f}"
    )

    print(
        f"Acceleration (a) = "
        f"{ci[3]:.6f}"
    )


# ============================================================
# 12. 输出成表格
# ============================================================

results = []

# Total
results.append({
    "Path": "Community → Evaluating AI",
    "Effect": "Total",
    "Estimate": total_effect,
    "CI Lower": total_ci[0],
    "CI Upper": total_ci[1]
})


# Direct
results.append({
    "Path": "Community → Evaluating AI",
    "Effect": "Direct",
    "Estimate": direct_effect,
    "CI Lower": direct_ci[0],
    "CI Upper": direct_ci[1]
})


# Indirect
mediator_names = {
    M1: "Value Efficacy",
    M2: "Skill Efficacy",
    M3: "Usage Efficacy"
}

for M in mediators:

    ci = indirect_ci_results[M]

    results.append({
        "Path": (
            f"Community → "
            f"{mediator_names[M]} → "
            f"Evaluating AI"
        ),
        "Effect": "Indirect",
        "Estimate": indirect_effects[M],
        "CI Lower": ci[0],
        "CI Upper": ci[1]
    })


results_df = pd.DataFrame(
    results
)


# ============================================================
# 13. 最终结果
# ============================================================

print("\n" + "=" * 70)
print("FINAL BCa BOOTSTRAP RESULTS")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

Data shape: (301, 28)
   Nmuber  Using AI  Evaluating AI  Access Motivation  Skill Motivation  \
0       1  3.000000       3.666667           3.666667               3.6   
1       2  4.000000       4.000000           4.333333               3.2   
2       3  3.000000       3.333333           3.666667               3.2   
3       4  4.000000       4.000000           3.000000               2.6   
4       5  2.666667       4.000000           4.333333               3.4   

   Usage Motivation  Using AI-1  Using AI-2  Using AI-3  Evaluating AI-1  ...  \
0              3.50           4           3           2                4  ...   
1              3.75           4           4           4                4  ...   
2              3.25           4           2           3                3  ...   
3              3.25           4           4           4                4  ...   
4              4.00           2           2           4                4  ...   

   Skill Motivation-4  Skill Motivation-

In [7]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import norm


# ============================================================
# 1. 读取数据
# ============================================================

file_path = r"D:\作业合集\数字化市场分析\论文\数据\data.xlsx"

data = pd.read_excel(
    file_path,
    sheet_name="Total"
)

print("Data shape:", data.shape)
print(data.head())


# ============================================================
# 2. 定义变量
# ============================================================

X = "Community"
Y = "Using AI"

M1 = "Access Motivation"   # Value Efficacy
M2 = "Skill Motivation"    # Skill Efficacy
M3 = "Usage Motivation"    # Usage Efficacy

controls = ["Gender", "Year"]

variables = [
    X, Y, M1, M2, M3,
    "Gender", "Year"
]

# 只保留分析需要的变量
df = data[variables].copy()

# 删除缺失值
df = df.dropna().reset_index(drop=True)

print("\nFinal N:", len(df))


# ============================================================
# 3. 检查变量类型
# ============================================================

print("\nVariable types:")
print(df.dtypes)

print("\nCommunity distribution:")
print(df[X].value_counts().sort_index())

print("\nGender distribution:")
print(df["Gender"].value_counts().sort_index())

print("\nYear distribution:")
print(df["Year"].value_counts().sort_index())


# ============================================================
# 4. 如果 Gender / Year 是分类变量，进行 dummy coding
# ============================================================

# Community 如果本身已经是数值编码，则直接使用
# 如果 Community 是字符串，也会自动转换成 category

if not pd.api.types.is_numeric_dtype(df[X]):
    df[X] = pd.Categorical(df[X]).codes

# Gender
if not pd.api.types.is_numeric_dtype(df["Gender"]):
    gender_dummies = pd.get_dummies(
        df["Gender"],
        prefix="Gender",
        drop_first=True,
        dtype=float
    )
else:
    gender_dummies = df[["Gender"]].astype(float)

# Year
if not pd.api.types.is_numeric_dtype(df["Year"]):
    year_dummies = pd.get_dummies(
        df["Year"],
        prefix="Year",
        drop_first=True,
        dtype=float
    )
else:
    year_dummies = df[["Year"]].astype(float)

# 合并控制变量
control_df = pd.concat(
    [gender_dummies, year_dummies],
    axis=1
)

# 确保所有数据都是 numeric
df[X] = pd.to_numeric(df[X])
df[Y] = pd.to_numeric(df[Y])
df[M1] = pd.to_numeric(df[M1])
df[M2] = pd.to_numeric(df[M2])


# ============================================================
# 5. Step 1: Total Effect
#
# Community → Evaluating AI
#
# Y = cX + controls
# ============================================================

X_total = pd.concat(
    [
        df[[X]],
        control_df
    ],
    axis=1
)

X_total = sm.add_constant(X_total)

model_total = sm.OLS(
    df[Y],
    X_total
).fit()

total_effect = model_total.params[X]

print("\n" + "=" * 70)
print("TOTAL EFFECT")
print("=" * 70)

print("Total effect (c):", total_effect)
print("p-value:", model_total.pvalues[X])


# ============================================================
# 6. Step 2: Direct Effect + a paths
#
# M = aX + controls
#
# Y = c'X + bM + controls
# ============================================================

mediators = [M1, M2, M3]

a_paths = {}
b_paths = {}
indirect_effects = {}

# ---------- a paths ----------

for M in mediators:

    X_m = pd.concat(
        [
            df[[X]],
            control_df
        ],
        axis=1
    )

    X_m = sm.add_constant(X_m)

    model_a = sm.OLS(
        df[M],
        X_m
    ).fit()

    a_paths[M] = model_a.params[X]


# ---------- b paths + direct effect ----------

X_direct = pd.concat(
    [
        df[[X, M1, M2, M3]],
        control_df
    ],
    axis=1
)

X_direct = sm.add_constant(X_direct)

model_direct = sm.OLS(
    df[Y],
    X_direct
).fit()

direct_effect = model_direct.params[X]

for M in mediators:
    b_paths[M] = model_direct.params[M]

    indirect_effects[M] = (
        a_paths[M] * b_paths[M]
    )


# ============================================================
# 7. 打印原始点估计
# ============================================================

print("\n" + "=" * 70)
print("MEDIATION RESULTS")
print("=" * 70)

print("\nTotal effect:")
print(f"c = {total_effect:.6f}")

print("\nDirect effect:")
print(f"c' = {direct_effect:.6f}")

print("\nPaths:")

for M in mediators:

    print(
        f"\n{X} → {M}: "
        f"a = {a_paths[M]:.6f}"
    )

    print(
        f"{M} → {Y}: "
        f"b = {b_paths[M]:.6f}"
    )

    print(
        f"Indirect effect = a × b = "
        f"{indirect_effects[M]:.6f}"
    )


# ============================================================
# 8. Bootstrap
# ============================================================

N_BOOT = 5000

rng = np.random.default_rng(42)

boot_total = []
boot_direct = []

boot_indirect = {
    M1: [],
    M2: [],
    M3: []
}

n = len(df)

print("\n" + "=" * 70)
print("BOOTSTRAP")
print("=" * 70)

print(f"Bootstrap samples: {N_BOOT}")

for i in range(N_BOOT):

    # bootstrap sampling
    indices = rng.integers(
        low=0,
        high=n,
        size=n
    )

    boot_df = df.iloc[
        indices
    ].reset_index(drop=True)

    # 控制变量同步抽样
    boot_control_df = control_df.iloc[
        indices
    ].reset_index(drop=True)

    # --------------------------------------------------------
    # Total effect
    # --------------------------------------------------------

    X_t = pd.concat(
        [
            boot_df[[X]],
            boot_control_df
        ],
        axis=1
    )

    X_t = sm.add_constant(X_t)

    try:

        model_t = sm.OLS(
            boot_df[Y],
            X_t
        ).fit()

        c = model_t.params[X]

        boot_total.append(c)

    except:
        continue


    # --------------------------------------------------------
    # a paths
    # --------------------------------------------------------

    boot_a = {}

    try:

        for M in mediators:

            X_a = pd.concat(
                [
                    boot_df[[X]],
                    boot_control_df
                ],
                axis=1
            )

            X_a = sm.add_constant(X_a)

            model_a = sm.OLS(
                boot_df[M],
                X_a
            ).fit()

            boot_a[M] = model_a.params[X]


        # ----------------------------------------------------
        # Direct + b paths
        # ----------------------------------------------------

        X_b = pd.concat(
            [
                boot_df[[X, M1, M2, M3]],
                boot_control_df
            ],
            axis=1
        )

        X_b = sm.add_constant(X_b)

        model_b = sm.OLS(
            boot_df[Y],
            X_b
        ).fit()

        c_prime = model_b.params[X]

        boot_direct.append(c_prime)


        # ----------------------------------------------------
        # indirect effects
        # ----------------------------------------------------

        for M in mediators:

            b = model_b.params[M]

            indirect = (
                boot_a[M] * b
            )

            boot_indirect[M].append(
                indirect
            )

    except:
        continue


# ============================================================
# 9. BCa Bootstrap 95% CI
# ============================================================

def bca_ci(
    bootstrap_values,
    original_estimate,
    data,
    statistic_func,
    alpha=0.05
):
    """
    BCa Bootstrap Confidence Interval

    Parameters
    ----------
    bootstrap_values : array-like
        Bootstrap estimates

    original_estimate : float
        Original sample estimate

    data : DataFrame
        Original dataset

    statistic_func : function
        Function that calculates the statistic from a dataset

    alpha : float
        Significance level, default = 0.05

    Returns
    -------
    lower, upper, z0, acceleration
    """

    bootstrap_values = np.asarray(
        bootstrap_values,
        dtype=float
    )

    bootstrap_values = bootstrap_values[
        np.isfinite(bootstrap_values)
    ]

    # --------------------------------------------------------
    # Step 1: Bias correction z0
    # --------------------------------------------------------

    proportion_less = np.mean(
        bootstrap_values < original_estimate
    )

    # 防止 norm.ppf(0) 或 norm.ppf(1)
    eps = 1e-10

    proportion_less = np.clip(
        proportion_less,
        eps,
        1 - eps
    )

    z0 = norm.ppf(
        proportion_less
    )


    # --------------------------------------------------------
    # Step 2: Jackknife estimates
    # --------------------------------------------------------

    jackknife_values = []

    for i in range(len(data)):

        jackknife_data = data.drop(
            data.index[i]
        ).reset_index(drop=True)

        try:

            estimate = statistic_func(
                jackknife_data
            )

            if np.isfinite(estimate):
                jackknife_values.append(
                    estimate
                )

        except:
            continue

    jackknife_values = np.asarray(
        jackknife_values,
        dtype=float
    )

    jackknife_values = jackknife_values[
        np.isfinite(jackknife_values)
    ]


    # --------------------------------------------------------
    # Step 3: Acceleration parameter
    #
    # a =
    # Σ(θ̄jack - θi)^3
    # --------------------------------
    # 6 [Σ(θ̄jack - θi)^2]^(3/2)
    # --------------------------------------------------------

    jack_mean = np.mean(
        jackknife_values
    )

    numerator = np.sum(
        (jack_mean - jackknife_values) ** 3
    )

    denominator = (
        6 *
        (
            np.sum(
                (jack_mean - jackknife_values) ** 2
            )
            ** 1.5
        )
    )

    if denominator == 0:
        acceleration = 0.0
    else:
        acceleration = (
            numerator / denominator
        )


    # --------------------------------------------------------
    # Step 4: Adjusted alpha levels
    # --------------------------------------------------------

    z_alpha_low = norm.ppf(
        alpha / 2
    )

    z_alpha_high = norm.ppf(
        1 - alpha / 2
    )

    # BCa adjusted probabilities
    adjusted_low = norm.cdf(
        z0
        +
        (
            z0 + z_alpha_low
        )
        /
        (
            1
            -
            acceleration
            *
            (
                z0 + z_alpha_low
            )
        )
    )

    adjusted_high = norm.cdf(
        z0
        +
        (
            z0 + z_alpha_high
        )
        /
        (
            1
            -
            acceleration
            *
            (
                z0 + z_alpha_high
            )
        )
    )

    # 防止概率超出 [0,1]
    adjusted_low = np.clip(
        adjusted_low,
        0,
        1
    )

    adjusted_high = np.clip(
        adjusted_high,
        0,
        1
    )


    # --------------------------------------------------------
    # Step 5: BCa confidence interval
    # --------------------------------------------------------

    lower = np.quantile(
        bootstrap_values,
        adjusted_low
    )

    upper = np.quantile(
        bootstrap_values,
        adjusted_high
    )

    return (
        lower,
        upper,
        z0,
        acceleration
    )


# ============================================================
# 10. 定义每个效应的 statistic function
# ============================================================

def statistic_total(data):

    X_t = pd.concat(
        [
            data[[X]],
            control_df.loc[
                data.index
            ].reset_index(drop=True)
        ],
        axis=1
    )

    X_t = sm.add_constant(X_t)

    model = sm.OLS(
        data[Y].values,
        X_t
    ).fit()

    return model.params[X]


# 为了 jackknife 时控制变量能够同步删除，
# 单独构建完整分析数据
analysis_df = pd.concat(
    [
        df[[X, Y, M1, M2, M3]],
        control_df
    ],
    axis=1
).reset_index(drop=True)


# ------------------------------------------------------------
# Total effect
# ------------------------------------------------------------

def total_statistic(d):

    X_t = d[
        [X] +
        list(control_df.columns)
    ]

    X_t = sm.add_constant(
        X_t,
        has_constant="add"
    )

    model = sm.OLS(
        d[Y],
        X_t
    ).fit()

    return model.params[X]


# ------------------------------------------------------------
# Direct effect
# ------------------------------------------------------------

def direct_statistic(d):

    X_b = d[
        [
            X,
            M1,
            M2,
            M3
        ]
        +
        list(control_df.columns)
    ]

    X_b = sm.add_constant(
        X_b,
        has_constant="add"
    )

    model = sm.OLS(
        d[Y],
        X_b
    ).fit()

    return model.params[X]


# ------------------------------------------------------------
# Indirect effect
# ------------------------------------------------------------

def indirect_statistic(M):

    def statistic(d):

        # a path
        X_a = d[
            [
                X
            ]
            +
            list(control_df.columns)
        ]

        X_a = sm.add_constant(
            X_a,
            has_constant="add"
        )

        model_a = sm.OLS(
            d[M],
            X_a
        ).fit()

        a = model_a.params[X]


        # b path
        X_b = d[
            [
                X,
                M1,
                M2,
                M3
            ]
            +
            list(control_df.columns)
        ]

        X_b = sm.add_constant(
            X_b,
            has_constant="add"
        )

        model_b = sm.OLS(
            d[Y],
            X_b
        ).fit()

        b = model_b.params[M]

        return a * b

    return statistic


# ============================================================
# 11. 计算 BCa CI
# ============================================================

print("\n" + "=" * 70)
print("BCa CONFIDENCE INTERVAL")
print("=" * 70)


# ------------------------------------------------------------
# Total effect
# ------------------------------------------------------------

total_ci = bca_ci(
    bootstrap_values=boot_total,
    original_estimate=total_effect,
    data=analysis_df,
    statistic_func=total_statistic
)

print(
    "\nTotal effect BCa CI:"
)

print(
    f"Estimate = {total_effect:.6f}"
)

print(
    f"95% BCa CI = "
    f"[{total_ci[0]:.6f}, "
    f"{total_ci[1]:.6f}]"
)

print(
    f"Bias correction (z0) = "
    f"{total_ci[2]:.6f}"
)

print(
    f"Acceleration (a) = "
    f"{total_ci[3]:.6f}"
)


# ------------------------------------------------------------
# Direct effect
# ------------------------------------------------------------

direct_ci = bca_ci(
    bootstrap_values=boot_direct,
    original_estimate=direct_effect,
    data=analysis_df,
    statistic_func=direct_statistic
)

print(
    "\nDirect effect BCa CI:"
)

print(
    f"Estimate = {direct_effect:.6f}"
)

print(
    f"95% BCa CI = "
    f"[{direct_ci[0]:.6f}, "
    f"{direct_ci[1]:.6f}]"
)

print(
    f"Bias correction (z0) = "
    f"{direct_ci[2]:.6f}"
)

print(
    f"Acceleration (a) = "
    f"{direct_ci[3]:.6f}"
)


# ------------------------------------------------------------
# Indirect effects
# ------------------------------------------------------------

indirect_ci_results = {}

for M in mediators:

    ci = bca_ci(
        bootstrap_values=boot_indirect[M],
        original_estimate=indirect_effects[M],
        data=analysis_df,
        statistic_func=indirect_statistic(M)
    )

    indirect_ci_results[M] = ci

    print(
        f"\n{M}"
    )

    print(
        f"Indirect effect = "
        f"{indirect_effects[M]:.6f}"
    )

    print(
        f"95% BCa CI = "
        f"[{ci[0]:.6f}, "
        f"{ci[1]:.6f}]"
    )

    print(
        f"Bias correction (z0) = "
        f"{ci[2]:.6f}"
    )

    print(
        f"Acceleration (a) = "
        f"{ci[3]:.6f}"
    )


# ============================================================
# 12. 输出成表格
# ============================================================

results = []

# Total
results.append({
    "Path": "Community → Using AI",
    "Effect": "Total",
    "Estimate": total_effect,
    "CI Lower": total_ci[0],
    "CI Upper": total_ci[1]
})


# Direct
results.append({
    "Path": "Community → Using AI",
    "Effect": "Direct",
    "Estimate": direct_effect,
    "CI Lower": direct_ci[0],
    "CI Upper": direct_ci[1]
})


# Indirect
mediator_names = {
    M1: "Value Efficacy",
    M2: "Skill Efficacy",
    M3: "Usage Efficacy"
}

for M in mediators:

    ci = indirect_ci_results[M]

    results.append({
        "Path": (
            f"Community → "
            f"{mediator_names[M]} → "
            f"Using AI"
        ),
        "Effect": "Indirect",
        "Estimate": indirect_effects[M],
        "CI Lower": ci[0],
        "CI Upper": ci[1]
    })


results_df = pd.DataFrame(
    results
)


# ============================================================
# 13. 最终结果
# ============================================================

print("\n" + "=" * 70)
print("FINAL BCa BOOTSTRAP RESULTS")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

Data shape: (301, 28)
   Nmuber  Using AI  Evaluating AI  Access Motivation  Skill Motivation  \
0       1  3.000000       3.666667           3.666667               3.6   
1       2  4.000000       4.000000           4.333333               3.2   
2       3  3.000000       3.333333           3.666667               3.2   
3       4  4.000000       4.000000           3.000000               2.6   
4       5  2.666667       4.000000           4.333333               3.4   

   Usage Motivation  Using AI-1  Using AI-2  Using AI-3  Evaluating AI-1  ...  \
0              3.50           4           3           2                4  ...   
1              3.75           4           4           4                4  ...   
2              3.25           4           2           3                3  ...   
3              3.25           4           4           4                4  ...   
4              4.00           2           2           4                4  ...   

   Skill Motivation-4  Skill Motivation-